# Detection parameter tuning against the ablation ground truth

Searches the candidate-detection variables of the **full (baseline) model** to find values that reduce
false positives while increasing recall.

- **Same raw images** as `ablations/study_01`: raw commit `8b2c2d92d7fff68dd73483b87adf314cfd10c5f7`, cohort `df.iloc[900:1000]`, same before-image preference and analysis grid.
- **Ground truth**: the 66 unique landslide sites pooled from all six labelled ablation variants (embedded below).
- **Scoring**: candidates are deduplicated exactly like `analyze_ablation.py` (global greedy IoU ≥ 0.5 on chip boxes). A candidate is a TP when its detection box has IoU ≥ `MATCH_IOU` with a site member; recall is sites found / sites in the split.
- **Search**: multi-objective Optuna (maximise sites found, minimise false positives) over 14 variables, starting from the baseline values.
- **Overfitting guard**: incidents are split into tuning and hold-out sets (spatially overlapping incidents stay together). Only the tuning split drives the search; the hold-out split decides whether the result generalises.

## How it works

1. IR-MAD change evidence (the expensive step) is computed **once** per incident with code copied verbatim from `ablation_baseline.ipynb`, and cached.
2. Each trial re-runs only the parameter-dependent steps (fusion, slope mask, SLIC, Bonferroni gate, hysteresis, blob filters, ranking). SLIC segmentations are cached per segmentation setting.
3. The baseline parameters are re-run first and compared with the ablation run's counts as a reproduction check.

## Kaggle setup

1. **Settings → Internet: On.** CPU session is enough; more cores make the search faster.
2. Optional: attach the `landslide-reproted` dataset to cross-check the cohort, and the `huggingface_token` secret (read access is enough; the raw dataset is public).
3. Adjust `N_TRIALS`, `TIMEOUT_HOURS`, `MATCH_IOU`, or `SELECTION_RULE` in the configuration cell, then **Run all**.
4. Results are written to `/kaggle/working/tuning_results/`. Re-running in the same session resumes from the cached evidence and the Optuna database.


In [1]:
%pip install -q "numpy>=1.24" "pandas>=2.0" "scipy>=1.10" "rasterio>=1.3" "scikit-image>=0.19,<2" "huggingface_hub>=0.23" "optuna>=3.4"

Note: you may need to restart the kernel to use updated packages.


## Configuration


In [2]:
from __future__ import annotations

import csv
import json
import math
import multiprocessing as mp
import os
import random
import shutil
import time
import traceback
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
import rasterio
from affine import Affine
from huggingface_hub import HfApi, hf_hub_download
from rasterio.transform import xy
from rasterio.warp import Resampling, calculate_default_transform, reproject, transform_bounds
from rasterio.windows import Window
from scipy.stats import chi2, norm
from skimage.filters import apply_hysteresis_threshold
from skimage.measure import label, regionprops
from skimage.morphology import remove_small_objects
from skimage.segmentation import slic

# Same raw imagery as ablations/study_01 (pinned commit recorded in every variant's summary.json).
RAW_REPO = "sasudo2/landslides"
RAW_COMMIT_SHA = "8b2c2d92d7fff68dd73483b87adf314cfd10c5f7"
RAW_ROOT = "raw_images/raw_incidents"
TARGET_CRS = "EPSG:4326"
SOURCE_CSV_NAME = "landslides_from_2018_to_2026.csv"
COHORT_START = 900
COHORT_STOP = 1000

WORKING = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
OUTPUT_DIR = WORKING / "tuning_results"
# Evidence and SLIC caches are large (~25 MB per incident) and stay outside the Kaggle output folder.
CACHE_DIR = Path("/tmp/tuning_cache")

EVIDENCE_WORKERS = 4                      # threads for download + IR-MAD (same as the ablation runs)
EVAL_WORKERS = max(1, os.cpu_count() or 1)  # processes for each parameter evaluation
N_TRIALS = 200
TIMEOUT_HOURS = 8.0
SAMPLER_SEED = 0
HOLDOUT_FRACTION = 1 / 3
SPLIT_SEED = 0

# A candidate is a true positive when its detection box has IoU >= MATCH_IOU with any member box of a
# ground-truth site. 0.2 matched only 67 of 7,229 manually labelled negatives across the six variants.
MATCH_IOU = 0.2
# Same global greedy dedup as analyze_ablation.py / deduplicate_candidates.py (on padded chip boxes).
DEDUP_IOU = 0.5
# "min_fp_at_baseline_recall": fewest false positives among trials that find >= baseline's sites.
# "max_f1" / "max_f2": best F-score (F2 weights recall over precision).
SELECTION_RULE = "min_fp_at_baseline_recall"

# Fixed settings copied from the ablation notebooks (IR-MAD, grid, chip padding).
SETTINGS = {
    "scale_m": 10.0,
    "max_iters": 20,
    "irmad_tol": 1e-3,
    "irmad_reg": 1e-6,
    "chip_pad_m": 300.0,
    "slic_min_segments": 50,
    "m_per_deg_lat": 111320.0,
}

# Baseline (full model) values of every tuned variable. The last three are new knobs whose baseline
# values reproduce the current pipeline exactly.
BASELINE_PARAMS = {
    "optical_weight": 0.5,          # geometric-mean weight of optical vs SAR confidence
    "min_slope_deg": 20.0,
    "target_superpixel_m": 30.0,
    "slic_compactness": 0.2,
    "high_percentile": 99.7,
    "low_percentile": 93.0,
    "alpha": 0.05,                  # Bonferroni-corrected per superpixel test
    "min_blob_area_m2": 3000.0,
    "max_blob_area_m2": 2000000.0,
    "max_elongation": 5.0,
    "min_mask_objects_px": 5,
    "max_candidates": 100,
    "min_severity": 0.0,            # new: drop candidates whose mean confidence is below this
    "score_mode": "area_x_severity",  # new: ranking used before the max_candidates cap
}

SEARCH_SPACE = {
    "optical_weight": ("categorical", [0.3, 0.4, 0.5, 0.6, 0.7]),
    "min_slope_deg": ("categorical", [10.0, 15.0, 20.0, 25.0, 30.0]),
    "target_superpixel_m": ("categorical", [20.0, 30.0, 40.0, 60.0]),
    "slic_compactness": ("categorical", [0.05, 0.1, 0.2, 0.5, 1.0]),
    "high_percentile": ("float", 98.0, 99.95, False),
    "low_percentile": ("float", 80.0, 99.5, False),
    "alpha": ("float", 1e-4, 0.2, True),
    "min_blob_area_m2": ("float", 500.0, 20000.0, True),
    "max_blob_area_m2": ("float", 1e5, 5e6, True),
    "max_elongation": ("float", 2.0, 10.0, False),
    "min_mask_objects_px": ("int", 1, 50, False),
    "max_candidates": ("int", 5, 100, False),
    "min_severity": ("float", 0.0, 0.99, False),
    "score_mode": ("categorical", ["area_x_severity", "severity", "sqrt_area_x_severity"]),
}
# SLIC output depends only on these, so segmentations are cached per combination.
SEGMENT_KEYS = ("optical_weight", "min_slope_deg", "target_superpixel_m", "slic_compactness")


def resolve_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("huggingface_token")
    except Exception:
        return None  # the raw dataset is public; a token only raises rate limits


HF_TOKEN = resolve_hf_token()
API = HfApi(token=HF_TOKEN)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(CACHE_DIR / "evidence").mkdir(parents=True, exist_ok=True)
(CACHE_DIR / "segments").mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore", category=optuna.exceptions.ExperimentalWarning)
# scikit-image >= 0.26 deprecates min_size; behaviour is unchanged and the ablation runs use the same call.
warnings.filterwarnings("ignore", message="Parameter `min_size` is deprecated", category=FutureWarning)
print(f"Outputs: {OUTPUT_DIR}\nCache: {CACHE_DIR}\nEvaluation workers: {EVAL_WORKERS}")

Outputs: /kaggle/working/tuning_results
Cache: /tmp/tuning_cache
Evaluation workers: 4


## Ground truth and cohort


In [3]:
# Ground truth: the 66 unique landslide sites pooled from all six ablation variants
# (ablations/study_01/ablation_results/ablation_sites.csv). Each member is the unpadded detection box
# (candidate_metadata.csv) of a candidate labelled "landslide". Site 56 has no metadata rows, so its
# label boxes are used with the 300 m chip padding removed.
# (site_id, site_incident, variant, member_incident, candidate_id, lon_min, lat_min, lon_max, lat_max)
GT_MEMBERS = [
    (1, 74290, 'optical_only', 74290, 39, 86.7428051, 27.0958562, 86.7438146, 27.0963951),
    (1, 74290, 'sar_only', 74290, 2, 86.7423004, 27.0981019, 86.7451268, 27.1003477),
    (2, 74290, 'optical_only', 74290, 13, 86.7166606, 27.0945087, 86.7200927, 27.095946),
    (3, 74290, 'no_bonferroni', 74290, 33, 86.7292786, 27.1104986, 86.730389, 27.1113071),
    (4, 74293, 'baseline', 74394, 58, 86.7123218, 27.7306963, 86.7130323, 27.731954),
    (4, 74293, 'no_slic', 74293, 40, 86.7127278, 27.7259353, 86.7132353, 27.7268336),
    (4, 74293, 'no_bonferroni', 74293, 48, 86.7146563, 27.724498, 86.7156713, 27.7257556),
    (4, 74293, 'no_bonferroni', 74394, 53, 86.7122203, 27.7306963, 86.7130323, 27.731954),
    (5, 74293, 'baseline', 74394, 68, 86.7111038, 27.7067114, 86.7116113, 27.7080589),
    (5, 74293, 'no_slic', 74293, 47, 86.7098858, 27.7104843, 86.7105963, 27.711203),
    (5, 74293, 'no_bonferroni', 74394, 62, 86.7111038, 27.7067114, 86.7116113, 27.7080589),
    (6, 74306, 'sar_only', 74306, 16, 83.5124516, 28.5441717, 83.5159301, 28.5457887),
    (7, 74306, 'sar_only', 74306, 23, 83.4906598, 28.5806431, 83.4929106, 28.5830686),
    (8, 74306, 'baseline', 74306, 73, 83.4713234, 28.5442615, 83.4723465, 28.5447107),
    (8, 74306, 'no_bonferroni', 74306, 66, 83.4713234, 28.5442615, 83.4723465, 28.5447107),
    (9, 74309, 'no_bonferroni', 74309, 14, 82.4327358, 27.9268288, 82.4336513, 27.9277271),
    (10, 74310, 'baseline', 74310, 2, 86.6075989, 27.4268901, 86.6107362, 27.4297647),
    (10, 74310, 'optical_only', 74310, 1, 86.6067892, 27.4262612, 86.6109386, 27.4296748),
    (10, 74310, 'optical_only', 74310, 16, 86.6053724, 27.4310223, 86.6071941, 27.4319206),
    (10, 74310, 'no_slope', 74310, 3, 86.6075989, 27.4266206, 86.611141, 27.4303036),
    (10, 74310, 'no_slic', 74310, 1, 86.6090157, 27.4268901, 86.6107362, 27.4296748),
    (10, 74310, 'no_bonferroni', 74310, 2, 86.6075989, 27.4268901, 86.6107362, 27.4297647),
    (11, 74310, 'baseline', 74310, 5, 86.6146831, 27.4140442, 86.6169095, 27.415751),
    (11, 74310, 'optical_only', 74310, 7, 86.6146831, 27.4138646, 86.6167071, 27.4159307),
    (11, 74310, 'no_slope', 74310, 10, 86.6147843, 27.4140442, 86.6167071, 27.415751),
    (11, 74310, 'no_slope', 74310, 23, 86.617112, 27.4195239, 86.6182252, 27.4215002),
    (11, 74310, 'no_slic', 74310, 4, 86.6148855, 27.4143137, 86.6159987, 27.4154815),
    (11, 74310, 'no_bonferroni', 74310, 4, 86.6146831, 27.4140442, 86.6169095, 27.415751),
    (12, 74310, 'no_slic', 74310, 7, 86.5987942, 27.4351545, 86.599705, 27.435963),
    (13, 74310, 'baseline', 74310, 12, 86.5775416, 27.4436885, 86.5789584, 27.4447665),
    (13, 74310, 'baseline', 74310, 14, 86.5744043, 27.4466529, 86.5757199, 27.4480004),
    (13, 74310, 'no_bonferroni', 74310, 14, 86.5742019, 27.4447665, 86.5755175, 27.4459343),
    (14, 74310, 'baseline', 74310, 16, 86.6093193, 27.4184459, 86.6104325, 27.4194341),
    (14, 74310, 'no_bonferroni', 74310, 19, 86.6093193, 27.4184459, 86.6104325, 27.4194341),
    (15, 74310, 'optical_only', 74310, 19, 86.5648912, 27.4285969, 86.5662068, 27.4296748),
    (15, 74310, 'optical_only', 74310, 26, 86.5615515, 27.4292257, 86.5624623, 27.4300342),
    (16, 74312, 'no_bonferroni', 74312, 8, 87.3314807, 26.9897836, 87.3329932, 26.9906819),
    (17, 74316, 'baseline', 74316, 13, 83.2440197, 28.376897, 83.2453467, 28.3782445),
    (18, 74321, 'optical_only', 74321, 4, 82.4526527, 28.7059991, 82.4554183, 28.7075262),
    (18, 74321, 'optical_only', 74321, 24, 82.4549062, 28.7077059, 82.4556232, 28.7090533),
    (19, 74321, 'optical_only', 74321, 10, 82.4422046, 28.703933, 82.4436386, 28.7056398),
    (20, 74329, 'baseline', 74329, 5, 86.7523174, 27.1111859, 86.7553447, 27.113252),
    (20, 74329, 'sar_only', 74329, 14, 86.7539319, 27.1119944, 86.7550419, 27.1133418),
    (21, 74389, 'baseline', 74389, 26, 80.8899691, 29.2255092, 80.8913069, 29.2262279),
    (21, 74389, 'optical_only', 74389, 28, 80.8892487, 29.2249703, 80.892336, 29.2263177),
    (21, 74389, 'no_slope', 74389, 40, 80.890072, 29.2255092, 80.891204, 29.2261381),
    (21, 74389, 'no_bonferroni', 74389, 26, 80.8899691, 29.2255092, 80.8913069, 29.2262279),
    (22, 74400, 'optical_only', 74400, 28, 80.4973866, 29.1907503, 80.4992386, 29.1922774),
    (22, 74400, 'sar_only', 74400, 30, 80.4997531, 29.1921876, 80.500782, 29.1938045),
    (23, 74400, 'baseline', 74400, 31, 80.4365791, 29.2017097, 80.4372993, 29.202608),
    (24, 74408, 'baseline', 74408, 32, 80.6139116, 29.540538, 80.6148407, 29.5414363),
    (25, 74408, 'baseline', 74408, 38, 80.5947108, 29.5486228, 80.5957431, 29.5491618),
    (26, 74408, 'no_slope', 74408, 60, 80.6319769, 29.5252667, 80.6326996, 29.5259853),
    (27, 74428, 'sar_only', 74428, 5, 81.6881497, 28.8537094, 81.6889705, 28.8546077),
    (28, 74441, 'baseline', 74441, 2, 84.5652518, 27.8325804, 84.570025, 27.8392279),
    (28, 74441, 'no_slic', 74441, 6, 84.5705328, 27.8347363, 84.5714469, 27.8370719),
    (28, 74441, 'no_slic', 74441, 9, 84.5673845, 27.8372516, 84.5688063, 27.8381499),
    (29, 74441, 'baseline', 74441, 12, 84.5859699, 27.7798495, 84.5875948, 27.7816461),
    (29, 74441, 'optical_only', 74441, 17, 84.5864777, 27.780119, 84.5874933, 27.7813766),
    (29, 74441, 'no_slope', 74441, 17, 84.5859699, 27.7797597, 84.5875948, 27.7813766),
    (29, 74441, 'no_slic', 74441, 8, 84.5864777, 27.7802088, 84.5872901, 27.7813766),
    (29, 74441, 'no_bonferroni', 74441, 14, 84.5859699, 27.7798495, 84.5875948, 27.7816461),
    (30, 74441, 'optical_only', 74441, 13, 84.495582, 27.805631, 84.4980194, 27.8067989),
    (31, 74441, 'optical_only', 74441, 15, 84.5732749, 27.7918869, 84.5755093, 27.7935038),
    (31, 74441, 'optical_only', 74441, 18, 84.5704313, 27.7933242, 84.5719547, 27.7945818),
    (31, 74441, 'sar_only', 74441, 13, 84.5718531, 27.7930547, 84.5737827, 27.7945818),
    (32, 74441, 'optical_only', 74441, 14, 84.4963945, 27.8375211, 84.4985272, 27.8385991),
    (33, 74565, 'optical_only', 74565, 2, 84.9262748, 28.0045244, 84.9301415, 28.0077583),
    (33, 74565, 'optical_only', 74565, 5, 84.925359, 28.0028176, 84.9268854, 28.0047939),
    (34, 74565, 'baseline', 74565, 8, 84.9417416, 28.0027278, 84.9424539, 28.0039854),
    (34, 74565, 'no_bonferroni', 74565, 7, 84.9415381, 28.0027278, 84.9424539, 28.0041651),
    (35, 74572, 'sar_only', 74572, 20, 87.3767561, 26.9987667, 87.3778653, 26.9998447),
    (36, 74582, 'baseline', 74582, 3, 87.8287123, 27.1291416, 87.8309333, 27.1307586),
    (36, 74582, 'no_slic', 74582, 1, 87.8284094, 27.1292314, 87.8305295, 27.1310281),
    (36, 74582, 'no_bonferroni', 74582, 6, 87.8287123, 27.1291416, 87.8309333, 27.1307586),
    (37, 74582, 'baseline', 74582, 15, 87.7669275, 27.1599537, 87.768038, 27.1607622),
    (37, 74582, 'no_slope', 74582, 5, 87.7653122, 27.1591452, 87.7678361, 27.1611215),
    (38, 74582, 'baseline', 74582, 14, 87.8257846, 27.1585164, 87.8266932, 27.1593249),
    (39, 74582, 'baseline', 74582, 19, 87.8205349, 27.1328247, 87.8217463, 27.1334535),
    (40, 74587, 'sar_only', 74587, 1, 85.6356631, 27.5256742, 85.6386011, 27.52792),
    (40, 74587, 'sar_only', 74587, 8, 85.6318133, 27.5237877, 85.6325224, 27.5247759),
    (40, 74587, 'no_slope', 74587, 8, 85.64083, 27.5275607, 85.6417418, 27.5286386),
    (41, 74587, 'no_slic', 74587, 1, 85.6439706, 27.5223504, 85.645085, 27.5232488),
    (42, 74587, 'sar_only', 74587, 6, 85.6529873, 27.5281895, 85.6544057, 27.5289081),
    (43, 74638, 'baseline', 74638, 1, 86.2683438, 27.3211676, 86.2742093, 27.3255693),
    (43, 74638, 'optical_only', 74638, 1, 86.2684449, 27.3211676, 86.2756251, 27.3266473),
    (43, 74638, 'no_slope', 74638, 1, 86.2683438, 27.3213473, 86.2732991, 27.324671),
    (43, 74638, 'no_slope', 74638, 4, 86.2726923, 27.3231439, 86.2742093, 27.3252998),
    (43, 74638, 'no_bonferroni', 74638, 1, 86.2683438, 27.3211676, 86.2742093, 27.3255693),
    (43, 74638, 'no_bonferroni', 74638, 22, 86.2724901, 27.319371, 86.273198, 27.3201795),
    (44, 74638, 'baseline', 74638, 17, 86.2728946, 27.3415593, 86.2739059, 27.3424576),
    (45, 74780, 'no_slope', 74780, 35, 85.766412, 27.9826956, 85.7674289, 27.9848516),
    (45, 74780, 'no_slic', 74780, 27, 85.7666154, 27.9834143, 85.7673272, 27.9848516),
    (46, 74809, 'optical_only', 74809, 5, 85.1090475, 27.4686513, 85.1107686, 27.4731429),
    (46, 74809, 'sar_only', 74809, 1, 85.1123884, 27.4670343, 85.116033, 27.4755683),
    (46, 74809, 'no_slope', 74809, 1, 85.1116797, 27.4701784, 85.1168429, 27.4770954),
    (47, 74809, 'optical_only', 74809, 7, 85.1013533, 27.4679327, 85.1033781, 27.46937),
    (48, 74836, 'baseline', 74836, 1, 85.5772164, 27.5971325, 85.5787369, 27.5983003),
    (48, 74836, 'no_slope', 74836, 12, 85.5772164, 27.5971325, 85.5787369, 27.5983003),
    (48, 74836, 'no_slic', 74836, 1, 85.5773178, 27.5973122, 85.5784328, 27.5983902),
    (49, 74836, 'optical_only', 74836, 8, 85.5721479, 27.6089004, 85.5727562, 27.6104275),
    (50, 74836, 'no_slope', 74836, 11, 85.5731616, 27.585724, 85.5745808, 27.586802),
    (51, 74946, 'baseline', 74946, 8, 86.3916873, 27.1436392, 86.393403, 27.1444477),
    (51, 74946, 'optical_only', 74946, 35, 86.3916873, 27.1436392, 86.3929993, 27.1442681),
    (51, 74946, 'sar_only', 74946, 10, 86.3888614, 27.1455257, 86.3902743, 27.1466935),
    (51, 74946, 'sar_only', 74946, 26, 86.3928984, 27.1448969, 86.3938067, 27.1457952),
    (51, 74946, 'no_slic', 74946, 2, 86.3916873, 27.1436392, 86.3929993, 27.1442681),
    (51, 74946, 'no_bonferroni', 74946, 9, 86.3916873, 27.1436392, 86.393403, 27.1444477),
    (52, 74946, 'sar_only', 74946, 12, 86.3827049, 27.1121085, 86.3843197, 27.113456),
    (53, 74946, 'optical_only', 74946, 14, 86.3655475, 27.1306137, 86.3673642, 27.1323205),
    (54, 74946, 'optical_only', 74946, 22, 86.3611068, 27.1357341, 86.3625198, 27.1373511),
    (55, 74953, 'baseline', 74953, 2, 86.4452924, 27.3087557, 86.4480223, 27.3113608),
    (56, 74962, 'optical_only', 74962, 1, 85.3241947, 27.5513439, 85.3279853, 27.5533741),
    (56, 74962, 'optical_only', 74962, 2, 85.3214907, 27.5523339, 85.3239513, 27.5536401),
    (57, 74972, 'baseline', 74972, 2, 85.6392952, 27.6923398, 85.6407157, 27.6937771),
    (57, 74972, 'sar_only', 74972, 7, 85.6375703, 27.6923398, 85.6384835, 27.6933279),
    (57, 74972, 'no_bonferroni', 74972, 3, 85.6391937, 27.6923398, 85.6407157, 27.6938669),
    (58, 74972, 'baseline', 74972, 6, 85.6536015, 27.7096772, 85.6546161, 27.7105755),
    (58, 74972, 'no_bonferroni', 74972, 6, 85.6536015, 27.7095873, 85.6546161, 27.7105755),
    (59, 74973, 'baseline', 74973, 1, 85.713865, 27.4396332, 85.7165979, 27.442418),
    (60, 74973, 'baseline', 74973, 3, 85.7227722, 27.4403519, 85.7244929, 27.4422383),
    (60, 74973, 'optical_only', 74973, 7, 85.722671, 27.4399925, 85.7244929, 27.4423282),
    (60, 74973, 'sar_only', 74973, 37, 85.7260112, 27.4384654, 85.7270234, 27.4396332),
    (60, 74973, 'no_slope', 74973, 10, 85.7227722, 27.4403519, 85.7244929, 27.4425977),
    (61, 74973, 'optical_only', 74973, 14, 85.7423074, 27.4567011, 85.7438256, 27.4582283),
    (61, 74973, 'optical_only', 74973, 27, 85.7374489, 27.4540062, 85.7383599, 27.455174),
    (61, 74973, 'sar_only', 74973, 15, 85.7415988, 27.4559825, 85.7433195, 27.4574198),
    (62, 74973, 'baseline', 74973, 24, 85.7387647, 27.4639775, 85.7394733, 27.4648758),
    (63, 74973, 'baseline', 74973, 31, 85.7303636, 27.433345, 85.731477, 27.4343332),
    (64, 74973, 'baseline', 74973, 34, 85.7628547, 27.4441248, 85.7640693, 27.4448434),
    (65, 75000, 'no_bonferroni', 75000, 23, 85.7709883, 27.9263715, 85.7739374, 27.9278088),
    (66, 75000, 'no_slic', 75000, 38, 85.792751, 27.9875465, 85.793768, 27.9885346),
]

# df.iloc[900:1000] of landslides_from_2018_to_2026.csv, in CSV order (the ablation cohort).
COHORT_IDS = [
    74962, 74969, 74970, 74971, 74972, 74973, 74974, 74975, 74982, 74985,
    74988, 74996, 74997, 74998, 74999, 75000, 75003, 75005, 75006, 75007,
    75019, 75024, 75030, 75037, 75043, 75044, 75048, 75049, 75050, 75081,
    75144, 75195, 74572, 74576, 74582, 74587, 74599, 74606, 74625, 74627,
    74638, 74643, 74645, 74688, 74729, 74747, 74809, 74836, 74890, 74946,
    74953, 74956, 74979, 75002, 74559, 74565, 74568, 74560, 74780, 74469,
    74465, 74456, 74441, 74442, 74446, 74449, 74443, 74398, 74399, 74400,
    74402, 74408, 74409, 74413, 74428, 74432, 74433, 74434, 74450, 74386,
    74389, 74394, 74336, 74337, 74306, 74308, 74309, 74310, 74312, 74313,
    74315, 74316, 74319, 74320, 74321, 74329, 74349, 74290, 74293, 74294,
]

# Counts from ablations/study_01 used to check that the baseline parameters reproduce the ablation run.
STUDY_REFERENCE = {"baseline_raw": 2582, "baseline_unique": 1079, "baseline_sites": 32, "failed_incidents": [74336]}

GT_BOXES = np.array([row[5:9] for row in GT_MEMBERS], dtype=np.float64)
GT_SITE_IDS = np.array([row[0] for row in GT_MEMBERS], dtype=np.int64)
SITE_INCIDENT = {row[0]: row[1] for row in GT_MEMBERS}
SITE_MEMBER_INCIDENTS = {}
for row in GT_MEMBERS:
    SITE_MEMBER_INCIDENTS.setdefault(row[0], {row[1]}).add(row[3])
SITE_MEMBER_INCIDENTS = {site: sorted(members) for site, members in SITE_MEMBER_INCIDENTS.items()}
SITE_INCIDENT_COUNTS = {}
for incident_id in SITE_INCIDENT.values():
    SITE_INCIDENT_COUNTS[incident_id] = SITE_INCIDENT_COUNTS.get(incident_id, 0) + 1
assert len(SITE_INCIDENT) == 66 and len(COHORT_IDS) == 100
print(f"Ground truth: {len(SITE_INCIDENT)} sites, {len(GT_MEMBERS)} member boxes, {len(SITE_INCIDENT_COUNTS)} incidents")

Ground truth: 66 sites, 132 member boxes, 28 incidents


## Detection helpers (verbatim from the ablation notebooks)


In [4]:
# Copied verbatim from ablation_baseline.ipynb so the evidence is identical to the ablation runs.
def norm01(x):
    a = np.nanmin(x)
    b = np.nanmax(x)
    if not np.isfinite(a) or not np.isfinite(b) or b <= a:
        return np.zeros_like(x, dtype=np.float32)
    return ((x - a) / (b - a)).astype(np.float32)


def wmean_cov(x, weights):
    weights = weights / np.sum(weights)
    mean = np.sum(x * weights[:, None], axis=0)
    centered = x - mean
    return mean, (centered * weights[:, None]).T @ centered


def cca_axes(sxx, syy, sxy, reg=None):
    if reg is None:
        reg = SETTINGS["irmad_reg"]
    lx = np.linalg.cholesky(sxx + reg * np.eye(sxx.shape[0]))
    ly = np.linalg.cholesky(syy + reg * np.eye(syy.shape[0]))
    inv_lx = np.linalg.inv(lx)
    inv_ly = np.linalg.inv(ly)
    u, _, vt = np.linalg.svd(inv_lx @ sxy @ inv_ly.T, full_matrices=False)
    return inv_lx.T @ u, inv_ly.T @ vt.T


def irmad(x, y, max_iters=None, tol=None):
    if max_iters is None:
        max_iters = SETTINGS["max_iters"]
    if tol is None:
        tol = SETTINGS["irmad_tol"]
    n, p = x.shape
    weights = np.ones(n, dtype=np.float64)
    previous = None
    chi = None
    for _ in range(max_iters):
        mx, sxx = wmean_cov(x, weights)
        my, syy = wmean_cov(y, weights)
        xc = x - mx
        yc = y - my
        sxy = (xc * (weights / np.sum(weights))[:, None]).T @ yc
        a, b = cca_axes(sxx, syy, sxy)
        mad = xc @ a[:, :p] - yc @ b[:, :p]
        std = np.std(mad, axis=0) + 1e-6
        chi = np.sum((mad / std[None, :]) ** 2, axis=1)
        no_change_p = 1.0 - chi2.cdf(chi, df=p)
        if previous is not None and np.mean(np.abs(no_change_p - previous)) < tol:
            break
        previous = no_change_p
        weights = np.clip(no_change_p, 1e-6, 1.0)
    if chi is None:
        raise ValueError("IR-MAD requires at least one iteration")
    return chi, chi2.cdf(chi, df=p)


def build_target_grid(ref_path, scale_m=None):
    if scale_m is None:
        scale_m = SETTINGS["scale_m"]
    with rasterio.open(ref_path) as src:
        _, bottom, _, top = transform_bounds(src.crs, TARGET_CRS, *src.bounds)
        mean_lat_rad = np.deg2rad((bottom + top) / 2.0)
        m_per_deg_lon = SETTINGS["m_per_deg_lat"] * max(np.cos(mean_lat_rad), 1e-6)
        transform, width, height = calculate_default_transform(
            src.crs,
            TARGET_CRS,
            src.width,
            src.height,
            *src.bounds,
            resolution=(scale_m / m_per_deg_lon, scale_m / SETTINGS["m_per_deg_lat"]),
        )
    if width is None or height is None:
        raise ValueError(f"Could not determine analysis-grid dimensions for {ref_path}")
    return transform, int(width), int(height), mean_lat_rad


def reproject_bands(path, indexes, transform, shape):
    out = np.zeros((len(indexes), *shape), dtype=np.float32)
    with rasterio.open(path) as src:
        for i, index in enumerate(indexes):
            source = src.read(index).astype(np.float32)
            source[src.read_masks(index) == 0] = np.nan
            reproject(
                source=source,
                destination=out[i],
                src_transform=src.transform,
                src_crs=src.crs,
                src_nodata=np.nan,
                dst_transform=transform,
                dst_crs=TARGET_CRS,
                dst_nodata=np.nan,
                resampling=Resampling.bilinear,
            )
            out[i][~np.isfinite(out[i])] = 0.0
    if not np.all(np.isfinite(out)):
        raise ValueError(f"Non-finite raster values in {path}")
    return out


def _looks_like_alpha_band(band):
    values = band[np.isfinite(band)]
    if values.size < 100:
        return False
    low, high = np.percentile(values, [1, 99])
    return bool(high - low < 2.0 or len(np.unique(values)) <= 8)


def select_rgb_band_indexes(path):
    with rasterio.open(path) as src:
        if src.count >= 4:
            return [1, 2, 3] if _looks_like_alpha_band(src.read(4)) else [3, 2, 1]
        if src.count >= 3:
            return [1, 2, 3]
        raise ValueError(f"Optical raster has too few bands ({src.count}) in {path}")


def load_optical(before_path, after_path, transform, shape):
    before_indexes = select_rgb_band_indexes(before_path)
    after_indexes = select_rgb_band_indexes(after_path)
    n_bands = min(len(before_indexes), len(after_indexes), 3)
    before_indexes = before_indexes[:n_bands]
    after_indexes = after_indexes[:n_bands]
    before = reproject_bands(before_path, before_indexes, transform, shape)
    after = reproject_bands(after_path, after_indexes, transform, shape)
    meta = {
        "before_rgb_band_indexes": before_indexes,
        "after_rgb_band_indexes": after_indexes,
    }
    return before, after, meta


def load_slope(path, transform, shape):
    return reproject_bands(path, [1], transform, shape)[0]


def load_sar(pre_path, post_path, transform, shape):
    return (
        reproject_bands(pre_path, [1, 2], transform, shape),
        reproject_bands(post_path, [1, 2], transform, shape),
    )


def crop_native(path, indexes, bbox):
    with rasterio.open(path) as src:
        left, bottom, right, top = transform_bounds(TARGET_CRS, src.crs, *bbox)
        win = src.window(left, bottom, right, top)
        c0 = max(0, int(np.floor(win.col_off)))
        r0 = max(0, int(np.floor(win.row_off)))
        c1 = min(src.width, int(np.ceil(win.col_off + win.width)))
        r1 = min(src.height, int(np.ceil(win.row_off + win.height)))
        if c1 <= c0 or r1 <= r0:
            raise ValueError(f"Candidate chip does not overlap {path}")
        win = Window.from_slices((r0, r1), (c0, c1))
        return src.read(indexes, window=win).astype(np.float32), src.window_transform(win), src.crs


def write_chip(path, array, transform, crs):
    data = array[None, ...] if array.ndim == 2 else array
    with rasterio.open(
        path,
        "w",
        driver="GTiff",
        dtype="float32",
        count=data.shape[0],
        height=data.shape[1],
        width=data.shape[2],
        crs=crs,
        transform=transform,
    ) as dst:
        dst.write(data.astype(np.float32))


class InsufficientCoverageError(ValueError):
    pass


def change_evidence(before, after, optical):
    bands, height, width = before.shape
    if optical:
        valid = np.all(before > 0, axis=0) & np.all(after > 0, axis=0)
    else:
        valid = np.all(before != 0, axis=0) & np.all(after != 0, axis=0)
    n_valid = int(np.count_nonzero(valid))
    if n_valid <= 100:
        modality = "Optical" if optical else "SAR"
        raise InsufficientCoverageError(
            f"{modality} has {n_valid} valid paired pixels; IR-MAD requires more than 100"
        )
    x = np.moveaxis(before, 0, -1)[valid]
    y = np.moveaxis(after, 0, -1)[valid]
    chi_values, confidence_values = irmad(x, y)
    if not np.all(np.isfinite(chi_values)):
        raise ValueError("IR-MAD produced non-finite change evidence")
    confidence = np.zeros((height, width), dtype=np.float32)
    chi = np.zeros((height, width), dtype=np.float64)
    df = np.zeros((height, width), dtype=np.float64)
    confidence[valid] = confidence_values
    chi[valid] = chi_values
    df[valid] = bands
    return confidence, chi, df, valid


def aggregate_segments(confidence, chi, df, terrain, segments):
    ids, inverse = np.unique(segments, return_inverse=True)
    inverse = inverse.reshape(segments.shape)
    active = inverse[terrain]
    counts = np.bincount(active, minlength=len(ids))
    conf_sum = np.bincount(active, weights=confidence[terrain], minlength=len(ids))
    chi_sum = np.bincount(active, weights=chi[terrain], minlength=len(ids))
    df_sum = np.bincount(active, weights=df[terrain], minlength=len(ids))
    conf_mean = np.divide(conf_sum, counts, out=np.zeros(len(ids)), where=counts > 0)
    z = np.divide(chi_sum - df_sum, np.sqrt(2.0 * df_sum), out=np.zeros(len(ids)), where=df_sum > 0)
    n_tests = max(int(np.count_nonzero(counts)), 1)
    return conf_mean[inverse].astype(np.float32), z[inverse].astype(np.float32), n_tests

## Pin raw inputs


In [5]:
TRANSIENT_STATUS_CODES = {408, 409, 412, 425, 429, 500, 502, 503, 504}


def _status_code(exc):
    return getattr(getattr(exc, "response", None), "status_code", None)


def is_transient_hub_error(exc):
    if _status_code(exc) in TRANSIENT_STATUS_CODES or isinstance(exc, (ConnectionError, TimeoutError)):
        return True
    message = str(exc).lower()
    markers = (
        "timeout", "timed out", "temporarily", "connection reset", "connection aborted",
        "rate limit", "too many requests", "server error", "bad gateway",
        "service unavailable", "gateway timeout",
    )
    return any(marker in message for marker in markers)


def retry_hub(action, description, attempts=5, base_delay=2.0):
    for attempt in range(1, attempts + 1):
        try:
            return action()
        except Exception as exc:
            if attempt >= attempts or not is_transient_hub_error(exc):
                raise
            delay = min(base_delay ** attempt, 60.0) + random.uniform(0.0, 1.0)
            print(f"{description} failed on attempt {attempt}/{attempts}: {exc}; retrying in {delay:.1f}s")
            time.sleep(delay)
    raise RuntimeError(f"Retry loop exhausted for {description}")


def locate_source_csv():
    input_root = Path("/kaggle/input")
    matches = sorted(input_root.rglob(SOURCE_CSV_NAME)) if input_root.is_dir() else []
    if (Path.cwd() / SOURCE_CSV_NAME).is_file():
        matches.append(Path.cwd() / SOURCE_CSV_NAME)
    return matches[0] if matches else None


source_csv = locate_source_csv()
if source_csv is not None:
    csv_ids = [int(value) for value in pd.read_csv(source_csv)["id"].iloc[COHORT_START:COHORT_STOP]]
    if csv_ids != COHORT_IDS:
        raise RuntimeError(f"{source_csv} iloc[{COHORT_START}:{COHORT_STOP}] differs from the ablation cohort")
    print(f"Cohort verified against {source_csv}")
else:
    print("Source CSV not attached; using the embedded ablation cohort IDs")

info = retry_hub(lambda: API.repo_info(RAW_REPO, repo_type="dataset", revision=RAW_COMMIT_SHA), "resolve raw commit")
if str(info.sha).lower() != RAW_COMMIT_SHA:
    raise RuntimeError(f"Raw revision resolved to {info.sha}, expected {RAW_COMMIT_SHA}")

raw_files = {
    entry.path
    for entry in retry_hub(
        lambda: list(API.list_repo_tree(
            RAW_REPO, repo_type="dataset", revision=RAW_COMMIT_SHA, path_in_repo=RAW_ROOT, recursive=True,
        )),
        "list raw files",
    )
    if hasattr(entry, "blob_id") or getattr(entry, "type", None) == "file"
}
MANDATORY_RAW_SUFFIXES = ("after.tif", "slope.tif", "sar_pre.tif", "sar_post.tif")
BEFORE_RAW_SUFFIXES = ("planet_before.tif", "gee_before.tif")  # same preference as the ablation runs
BEFORE_BY_INCIDENT = {}
missing = {}
for incident_id in COHORT_IDS:
    prefix = f"{RAW_ROOT}/incident_{incident_id}/incident_{incident_id}_"
    absent = [suffix for suffix in MANDATORY_RAW_SUFFIXES if prefix + suffix not in raw_files]
    before = next((suffix for suffix in BEFORE_RAW_SUFFIXES if prefix + suffix in raw_files), None)
    if absent or before is None:
        missing[incident_id] = absent + ([] if before else ["before"])
    else:
        BEFORE_BY_INCIDENT[incident_id] = before
if missing:
    raise RuntimeError(f"Raw inputs missing at {RAW_COMMIT_SHA}: {missing}")
print(f"Pinned {RAW_REPO}@{RAW_COMMIT_SHA}; all {len(COHORT_IDS)} cohort incidents have raw inputs")

Cohort verified against /kaggle/input/datasets/sanjayashrestha123/landslide-reproted/landslides_from_2018_to_2026.csv
Pinned sasudo2/landslides@8b2c2d92d7fff68dd73483b87adf314cfd10c5f7; all 100 cohort incidents have raw inputs


## Cache change evidence (one-off, slow)


In [6]:
def evidence_path(incident_id):
    return CACHE_DIR / "evidence" / f"incident_{incident_id}.npz"


def fetch_paths(incident_id, raw_dir):
    prefix = f"{RAW_ROOT}/incident_{incident_id}/incident_{incident_id}_"
    suffixes = {
        "before": BEFORE_BY_INCIDENT[incident_id], "after": "after.tif", "slope": "slope.tif",
        "sar_pre": "sar_pre.tif", "sar_post": "sar_post.tif",
    }
    return {
        key: retry_hub(
            lambda suffix=suffix: hf_hub_download(
                repo_id=RAW_REPO, repo_type="dataset", revision=RAW_COMMIT_SHA,
                filename=prefix + suffix, token=HF_TOKEN, cache_dir=str(raw_dir),
            ),
            f"download {prefix}{suffix}",
        )
        for key, suffix in suffixes.items()
    }


def build_evidence(incident_id):
    """Run the fused baseline up to the parameter-dependent steps and cache the intermediate arrays."""
    target = evidence_path(incident_id)
    if target.exists():
        return incident_id, "ok", ""
    raw_dir = CACHE_DIR / "raw" / f"incident_{incident_id}"
    try:
        paths = fetch_paths(incident_id, raw_dir)
        transform, width, height, latitude = build_target_grid(paths["before"])
        shape = (height, width)
        slope = load_slope(paths["slope"], transform, shape)
        before, after, _ = load_optical(paths["before"], paths["after"], transform, shape)
        conf_opt, chi_opt, df_opt, valid_opt = change_evidence(before, after, True)
        pre, post = load_sar(paths["sar_pre"], paths["sar_post"], transform, shape)
        conf_sar, chi_sar, df_sar, valid_sar = change_evidence(pre, post, False)
        valid = valid_opt & valid_sar
        if int(np.count_nonzero(valid)) <= 100:
            raise InsufficientCoverageError(
                f"Fused modalities have {int(np.count_nonzero(valid))} jointly valid paired pixels; "
                "detection requires more than 100"
            )
        chi = chi_opt + chi_sar
        chi[~valid] = 0.0
        df = df_opt + df_sar
        df[~valid] = 0.0
        m_per_deg_lon = SETTINGS["m_per_deg_lat"] * max(np.cos(latitude), 1e-6)
        px_area = abs(transform.a * transform.e) * m_per_deg_lon * SETTINGS["m_per_deg_lat"]
        staging = target.with_name(target.stem + ".partial.npz")
        np.savez(
            staging,
            conf_opt=conf_opt, conf_sar=conf_sar, chi=chi, df=df, valid=valid, slope=slope,
            slope_norm=norm01(slope),
            post_norm=np.stack([norm01(band) for band in after], axis=-1),
            transform=np.array(tuple(transform)[:6], dtype=np.float64),
            px_area=np.float64(px_area),
        )
        os.replace(staging, target)
        return incident_id, "ok", ""
    except Exception as exc:
        return incident_id, "failed", f"{type(exc).__name__}: {exc}"
    finally:
        shutil.rmtree(raw_dir, ignore_errors=True)


started = time.time()
EVIDENCE_STATUS = {}
with ThreadPoolExecutor(max_workers=EVIDENCE_WORKERS) as pool:
    futures = [pool.submit(build_evidence, incident_id) for incident_id in COHORT_IDS]
    for done, future in enumerate(as_completed(futures), 1):
        incident_id, status, error = future.result()
        EVIDENCE_STATUS[incident_id] = (status, error)
        if status != "ok" or done % 10 == 0:
            print(f"[{done}/{len(COHORT_IDS)}] incident {incident_id}: {status} {error}")
with (OUTPUT_DIR / "evidence_status.csv").open("w", newline="", encoding="utf-8") as handle:
    writer = csv.writer(handle)
    writer.writerow(["incident_id", "status", "error"])
    writer.writerows([incident_id, *EVIDENCE_STATUS[incident_id]] for incident_id in COHORT_IDS)

OK_IDS = [incident_id for incident_id in COHORT_IDS if EVIDENCE_STATUS[incident_id][0] == "ok"]
failed = sorted(set(COHORT_IDS) - set(OK_IDS))
print(f"Evidence ready for {len(OK_IDS)} incidents in {(time.time() - started) / 60:.1f} min; failed: {failed}")
if failed != STUDY_REFERENCE["failed_incidents"]:
    print(f"WARNING: the ablation study failed {STUDY_REFERENCE['failed_incidents']}, this run failed {failed}")

raw_images/raw_incidents/incident_74971/(…): reconstructing file:   0%|          |  0.00B / 2.77MB            

raw_images/raw_incidents/incident_74969/(…): reconstructing file:   0%|          |  0.00B / 2.77MB            

raw_images/raw_incidents/incident_74970/(…): reconstructing file:   0%|          |  0.00B / 5.72MB            

raw_images/raw_incidents/incident_74971/(…): downloading bytes:           |  0.00B            

raw_images/raw_incidents/incident_74962/(…): reconstructing file:   0%|          |  0.00B / 2.38MB            

raw_images/raw_incidents/incident_74969/(…): downloading bytes:           |  0.00B            

... (1,002 outputs left out)

raw_images/raw_incidents/incident_74293/(…): downloading bytes:           |  0.00B            

raw_images/raw_incidents/incident_74294/(…): reconstructing file:   0%|          |  0.00B / 13.6MB            

raw_images/raw_incidents/incident_74294/(…): downloading bytes:           |  0.00B            

[100/100] incident 74294: ok 
Evidence ready for 99 incidents in 6.6 min; failed: [74336]


## Parametric detector and scoring


In [7]:
def load_evidence(incident_id):
    with np.load(evidence_path(incident_id)) as data:
        return {key: data[key] for key in data.files}


def fused_confidence(evidence, optical_weight):
    optical, sar = evidence["conf_opt"], evidence["conf_sar"]
    if optical_weight == 0.5:
        # Identical to the ablation pipeline's geometric mean.
        confidence = np.power(np.prod(np.stack([optical, sar], axis=0), axis=0), 0.5)
    else:
        confidence = np.power(optical, optical_weight) * np.power(sar, 1.0 - optical_weight)
    confidence = confidence.astype(np.float32)
    confidence[~evidence["valid"]] = 0.0
    return confidence


def segment_key(params):
    return "_".join(f"{key}={params[key]:g}" for key in SEGMENT_KEYS)


def get_segments(incident_id, evidence, confidence, params):
    path = CACHE_DIR / "segments" / f"incident_{incident_id}" / f"{segment_key(params)}.npz"
    if path.exists():
        try:
            with np.load(path) as data:
                return data["segments"]
        except Exception:
            path.unlink(missing_ok=True)
    features = np.concatenate(
        [confidence[..., None], evidence["post_norm"], evidence["slope_norm"][..., None]], axis=-1
    )
    px_area = float(evidence["px_area"])
    pixels_per_segment = max(1, int(params["target_superpixel_m"] ** 2 / max(px_area, 1e-6)))
    n_segments = max(SETTINGS["slic_min_segments"], int(confidence.size / pixels_per_segment))
    segments = slic(
        features, n_segments=n_segments, compactness=params["slic_compactness"],
        start_label=1, channel_axis=-1,
    ).astype(np.int32)
    path.parent.mkdir(parents=True, exist_ok=True)
    staging = path.with_name(f"{path.stem}.{os.getpid()}.partial.npz")
    np.savez_compressed(staging, segments=segments)
    os.replace(staging, path)
    return segments


def candidate_score(area, severity, mode):
    if mode == "area_x_severity":
        return area * severity
    if mode == "sqrt_area_x_severity":
        return math.sqrt(area) * severity
    if mode == "severity":
        return severity
    raise ValueError(f"Unknown score_mode {mode!r}")


def detect_cached(incident_id, params):
    """The baseline fused detector (slope mask, SLIC, Bonferroni) with every variable taken from params."""
    evidence = load_evidence(incident_id)
    valid, slope = evidence["valid"], evidence["slope"]
    confidence = fused_confidence(evidence, params["optical_weight"])
    terrain = valid & (slope >= params["min_slope_deg"])
    confidence *= terrain
    segments = get_segments(incident_id, evidence, confidence, params)
    seg_conf, z, n_tests = aggregate_segments(confidence, evidence["chi"], evidence["df"], terrain, segments)
    z_crit = norm.ppf(1.0 - params["alpha"] / n_tests)
    valid_conf = seg_conf[terrain]
    if valid_conf.size == 0 or np.max(valid_conf) <= 0:
        return []
    high = np.percentile(valid_conf, params["high_percentile"])
    low = min(np.percentile(valid_conf, params["low_percentile"]), high)
    mask = apply_hysteresis_threshold(seg_conf, low, high) & terrain & (z >= z_crit)
    mask = remove_small_objects(mask, min_size=int(params["min_mask_objects_px"]))

    transform = Affine(*evidence["transform"])
    px_area = float(evidence["px_area"])
    height, width = mask.shape
    pad = max(1, int(round(SETTINGS["chip_pad_m"] / max(np.sqrt(px_area), 1e-6))))
    candidates = []
    for region in regionprops(label(mask), intensity_image=confidence):
        area = region.area * px_area
        if not params["min_blob_area_m2"] <= area <= params["max_blob_area_m2"]:
            continue
        elongation = (region.major_axis_length or 1) / max(region.minor_axis_length or 1, 1e-6)
        if elongation > params["max_elongation"]:
            continue
        severity = float(region.mean_intensity)
        if severity < params["min_severity"]:
            continue
        r0, c0, r1, c1 = region.bbox
        lon_min, lat_max = xy(transform, r0, c0, offset="ul")
        lon_max, lat_min = xy(transform, r1, c1, offset="ul")
        chip_lon_min, chip_lat_max = xy(transform, max(0, r0 - pad), max(0, c0 - pad), offset="ul")
        chip_lon_max, chip_lat_min = xy(transform, min(height, r1 + pad), min(width, c1 + pad), offset="ul")
        candidates.append({
            "incident": incident_id,
            "area_m2": float(area),
            "elongation": float(elongation),
            "severity": severity,
            "score": float(candidate_score(area, severity, params["score_mode"])),
            "box": (float(lon_min), float(lat_min), float(lon_max), float(lat_max)),
            "chip_box": (float(chip_lon_min), float(chip_lat_min), float(chip_lon_max), float(chip_lat_max)),
        })
    candidates.sort(key=lambda item: item["score"], reverse=True)
    candidates = candidates[:int(params["max_candidates"])]
    for rank, candidate in enumerate(candidates, 1):
        candidate["rank"] = rank
    return candidates


def _detect_worker(args):
    incident_id, params = args
    try:
        return incident_id, detect_cached(incident_id, params), None
    except Exception:
        return incident_id, [], traceback.format_exc()


_POOL = None


def run_detection(params, incident_ids):
    global _POOL
    if _POOL is None:
        _POOL = mp.get_context("fork").Pool(EVAL_WORKERS)
    candidates = []
    for incident_id, found, error in _POOL.imap_unordered(
        _detect_worker, [(incident_id, params) for incident_id in incident_ids]
    ):
        if error:
            raise RuntimeError(f"Detection failed for incident {incident_id}:\n{error}")
        candidates.extend(found)
    candidates.sort(key=lambda item: (item["incident"], item["rank"]))
    return candidates


def iou_one_to_many(box, boxes):
    if len(boxes) == 0:
        return np.zeros(0)
    left = np.maximum(box[0], boxes[:, 0])
    bottom = np.maximum(box[1], boxes[:, 1])
    right = np.minimum(box[2], boxes[:, 2])
    top = np.minimum(box[3], boxes[:, 3])
    intersection = np.clip(right - left, 0, None) * np.clip(top - bottom, 0, None)
    union = (box[2] - box[0]) * (box[3] - box[1]) + (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    union = union - intersection
    return np.divide(intersection, union, out=np.zeros_like(union), where=union > 0)


def deduplicate(candidates, threshold=None):
    """Global greedy dedup in (incident, rank) order, as analyze_ablation.py does before scoring."""
    threshold = DEDUP_IOU if threshold is None else threshold
    kept = []
    kept_boxes = np.empty((max(len(candidates), 1), 4))
    for candidate in sorted(candidates, key=lambda item: (item["incident"], item["rank"])):
        box = np.asarray(candidate["chip_box"], dtype=np.float64)
        if np.any(iou_one_to_many(box, kept_boxes[:len(kept)]) >= threshold):
            continue
        kept_boxes[len(kept)] = box
        kept.append(candidate)
    return kept


def match_sites(candidates):
    """Annotate each candidate with the ground-truth sites its detection box matches."""
    for candidate in candidates:
        overlaps = iou_one_to_many(np.asarray(candidate["box"], dtype=np.float64), GT_BOXES)
        candidate["sites"] = sorted({int(site) for site in GT_SITE_IDS[overlaps >= MATCH_IOU]})
    return candidates


def score_candidates(unique, incident_ids):
    incident_ids = set(incident_ids)
    rows = [candidate for candidate in unique if candidate["incident"] in incident_ids]
    sites_total = {site for site, incident in SITE_INCIDENT.items() if incident in incident_ids}
    found = set().union(*(candidate["sites"] for candidate in rows)) & sites_total if rows else set()
    tp = sum(bool(candidate["sites"]) for candidate in rows)
    precision = tp / len(rows) if rows else 0.0
    recall = len(found) / len(sites_total) if sites_total else 0.0

    def f_beta(beta):
        denominator = beta ** 2 * precision + recall
        return (1 + beta ** 2) * precision * recall / denominator if denominator else 0.0

    return {
        "n_incidents": len(incident_ids),
        "unique_candidates": len(rows),
        "tp": tp,
        "fp": len(rows) - tp,
        "precision": precision,
        "sites_found": len(found),
        "sites_total": len(sites_total),
        "recall": recall,
        "f1": f_beta(1.0),
        "f2": f_beta(2.0),
        "incidents_hit": len({candidate["incident"] for candidate in rows if candidate["sites"]}),
        "candidates_per_tp": len(rows) / tp if tp else math.inf,
    }


def evaluate(params, incident_ids):
    raw = run_detection(params, incident_ids)
    unique = match_sites(deduplicate(raw))
    return raw, unique

## Tuning/hold-out split and baseline reproduction


In [8]:
def incident_bounds(incident_id):
    with np.load(evidence_path(incident_id)) as data:
        height, width = data["valid"].shape
        transform = Affine(*data["transform"])
    left, top = transform * (0, 0)
    right, bottom = transform * (width, height)
    return min(left, right), min(bottom, top), max(left, right), max(bottom, top)


def spatial_groups(incident_ids):
    """Incidents whose rasters overlap, or that share a ground-truth site, must stay in the same split."""
    parent = {incident_id: incident_id for incident_id in incident_ids}

    def find(item):
        while parent[item] != item:
            parent[item] = parent[parent[item]]
            item = parent[item]
        return item

    bounds = {incident_id: incident_bounds(incident_id) for incident_id in incident_ids}
    ids = list(incident_ids)
    for i, first in enumerate(ids):
        a = bounds[first]
        for second in ids[i + 1:]:
            b = bounds[second]
            if min(a[2], b[2]) > max(a[0], b[0]) and min(a[3], b[3]) > max(a[1], b[1]):
                parent[find(first)] = find(second)
    for members in SITE_MEMBER_INCIDENTS.values():
        members = [incident_id for incident_id in members if incident_id in parent]
        for other in members[1:]:
            parent[find(other)] = find(members[0])
    groups = {}
    for incident_id in ids:
        groups.setdefault(find(incident_id), []).append(incident_id)
    return sorted((sorted(group) for group in groups.values()), key=lambda group: group[0])


def split_incidents(incident_ids, fraction, seed):
    """Stratified group split: hold out ~fraction of the sites and ~fraction of landslide-free incidents."""
    rng = random.Random(seed)
    groups = spatial_groups(incident_ids)
    sites_in = lambda group: sum(SITE_INCIDENT_COUNTS.get(incident_id, 0) for incident_id in group)
    with_sites = [group for group in groups if sites_in(group) > 0]
    without_sites = [group for group in groups if sites_in(group) == 0]
    rng.shuffle(with_sites)
    rng.shuffle(without_sites)
    holdout = []
    total_sites = sum(sites_in(group) for group in with_sites)
    held_sites = 0
    for group in with_sites:
        if held_sites + sites_in(group) <= fraction * total_sites + 1e-9 or held_sites == 0:
            holdout.extend(group)
            held_sites += sites_in(group)
    target_free = fraction * sum(len(group) for group in without_sites)
    held_free = 0
    for group in without_sites:
        if held_free + len(group) <= target_free + 1e-9:
            holdout.extend(group)
            held_free += len(group)
    holdout = sorted(holdout)
    tune = sorted(set(incident_ids) - set(holdout))
    return tune, holdout


TUNE_IDS, HOLDOUT_IDS = split_incidents(OK_IDS, HOLDOUT_FRACTION, SPLIT_SEED)
SPLITS = {"tune": TUNE_IDS, "holdout": HOLDOUT_IDS, "all": OK_IDS}
(OUTPUT_DIR / "split.json").write_text(json.dumps(SPLITS, indent=2) + "\n", encoding="utf-8")
for name, ids in SPLITS.items():
    n_sites = sum(SITE_INCIDENT_COUNTS.get(incident_id, 0) for incident_id in ids)
    print(f"{name:>8}: {len(ids)} incidents, {n_sites} ground-truth sites")

# Sanity check: the baseline parameters must reproduce the ablation baseline run.
started = time.time()
BASELINE_RAW, BASELINE_UNIQUE = evaluate(BASELINE_PARAMS, OK_IDS)
baseline_all = score_candidates(BASELINE_UNIQUE, OK_IDS)
print(f"Baseline reproduction ({time.time() - started:.0f}s, includes first SLIC pass):")
print(f"  raw candidates   {len(BASELINE_RAW):5d}  (ablation baseline: {STUDY_REFERENCE['baseline_raw']})")
print(f"  unique candidates{baseline_all['unique_candidates']:5d}  (ablation baseline: {STUDY_REFERENCE['baseline_unique']})")
print(f"  sites found      {baseline_all['sites_found']:5d}/{baseline_all['sites_total']}"
      f"  (ablation baseline labelled: {STUDY_REFERENCE['baseline_sites']}; spatial matching can add neighbours)")
if len(BASELINE_RAW) != STUDY_REFERENCE["baseline_raw"]:
    print("WARNING: baseline candidates differ from the ablation run; check package versions and raw inputs.")

    tune: 62 incidents, 44 ground-truth sites
 holdout: 37 incidents, 22 ground-truth sites
     all: 99 incidents, 66 ground-truth sites
Baseline reproduction (53s, includes first SLIC pass):
  raw candidates    2582  (ablation baseline: 2582)
  unique candidates 1084  (ablation baseline: 1079)
  sites found         39/66  (ablation baseline labelled: 32; spatial matching can add neighbours)


## Parameter search


In [9]:
def suggest_params(trial):
    params = {}
    for name, spec in SEARCH_SPACE.items():
        kind = spec[0]
        if kind == "categorical":
            params[name] = trial.suggest_categorical(name, spec[1])
        elif kind == "float":
            params[name] = trial.suggest_float(name, spec[1], spec[2], log=spec[3])
        else:
            params[name] = trial.suggest_int(name, spec[1], spec[2], log=spec[3])
    return params


def objective(trial):
    params = suggest_params(trial)
    _, unique = evaluate(params, TUNE_IDS)
    metrics = score_candidates(unique, TUNE_IDS)
    for key, value in metrics.items():
        trial.set_user_attr(key, float(value) if math.isfinite(value) else None)
    return metrics["sites_found"], metrics["fp"]


def progress(study, trial):
    if trial.state != optuna.trial.TrialState.COMPLETE:
        return
    attrs = trial.user_attrs
    print(
        f"trial {trial.number:4d}: sites {attrs['sites_found']:.0f}/{attrs['sites_total']:.0f} "
        f"FP {attrs['fp']:.0f} precision {100 * attrs['precision']:.1f}% "
        f"F1 {100 * attrs['f1']:.1f}% | Pareto size {len(study.best_trials)} | {trial.duration}"
    )


# Persistent storage lets an interrupted session resume the search from the same database.
study = optuna.create_study(
    study_name="detection_tuning",
    storage=f"sqlite:///{OUTPUT_DIR / 'optuna.db'}",
    load_if_exists=True,
    directions=["maximize", "minimize"],  # (ground-truth sites found, false-positive candidates)
    sampler=optuna.samplers.TPESampler(seed=SAMPLER_SEED, multivariate=True, n_startup_trials=30),
)
if not study.trials:
    study.enqueue_trial(BASELINE_PARAMS)  # trial 0 is the ablation baseline on the tuning split
completed = sum(trial.state == optuna.trial.TrialState.COMPLETE for trial in study.trials)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(
    objective,
    n_trials=max(0, N_TRIALS - completed),
    timeout=TIMEOUT_HOURS * 3600,
    callbacks=[progress],
    gc_after_trial=True,
)
print(f"Completed trials: {sum(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials)}")

[I 2026-11-05 04:53:45,887] A new study created in RDB with name: detection_tuning


trial    0: sites 27/44 FP 623 precision 4.7% F1 8.8% | Pareto size 1 | 0:00:04.208989
trial    1: sites 2/44 FP 23 precision 8.0% F1 5.8% | Pareto size 2 | 0:00:26.533728
trial    2: sites 20/44 FP 484 precision 4.3% F1 7.9% | Pareto size 3 | 0:00:33.076882
trial    3: sites 0/44 FP 30 precision 0.0% F1 0.0% | Pareto size 3 | 0:00:31.059566
trial    4: sites 18/44 FP 611 precision 3.0% F1 5.6% | Pareto size 3 | 0:00:34.543555
trial    5: sites 2/44 FP 100 precision 2.0% F1 2.7% | Pareto size 3 | 0:00:28.495996
trial    6: sites 1/44 FP 53 precision 1.9% F1 2.0% | Pareto size 3 | 0:00:33.612902
trial    7: sites 17/44 FP 932 precision 2.0% F1 3.8% | Pareto size 3 | 0:00:29.729579
trial    8: sites 18/44 FP 248 precision 7.5% F1 12.6% | Pareto size 4 | 0:00:31.758670
trial    9: sites 0/44 FP 0 precision 0.0% F1 0.0% | Pareto size 5 | 0:00:34.600405
trial   10: sites 1/44 FP 47 precision 2.1% F1 2.2% | Pareto size 5 | 0:00:25.475671
trial   11: sites 17/44 FP 456 precision 4.4% F1 7.9% 

## Select, validate on hold-out, and export


In [10]:
from IPython.display import display


def trial_table(study):
    rows = []
    pareto = {trial.number for trial in study.best_trials}
    for trial in study.trials:
        if trial.state != optuna.trial.TrialState.COMPLETE:
            continue
        rows.append({"trial": trial.number, "pareto": trial.number in pareto, **trial.params, **trial.user_attrs})
    return pd.DataFrame(rows)


def plain(value):
    if isinstance(value, dict):
        return {key: plain(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [plain(item) for item in value]
    if hasattr(value, "item"):
        value = value.item()
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


def select_trial(trials, baseline, rule):
    if rule == "min_fp_at_baseline_recall":
        eligible = trials[trials["sites_found"] >= baseline["sites_found"]]
        if eligible.empty:
            return select_trial(trials, baseline, "max_f2")
        return eligible.sort_values(["fp", "sites_found", "precision"], ascending=[True, False, False]).iloc[0]
    if rule in ("max_f1", "max_f2"):
        metric = rule.split("_")[1]
        return trials.sort_values([metric, "sites_found"], ascending=[False, False]).iloc[0]
    raise ValueError(f"Unknown SELECTION_RULE {rule!r}")


TRIALS = trial_table(study)
TRIALS.to_csv(OUTPUT_DIR / "trials.csv", index=False)
TRIALS[TRIALS["pareto"]].sort_values("sites_found").to_csv(OUTPUT_DIR / "pareto.csv", index=False)
baseline_tune = score_candidates(BASELINE_UNIQUE, TUNE_IDS)

# Evaluate the pick of every rule on the untouched hold-out incidents and on the full cohort.
picks = {"baseline": BASELINE_PARAMS}
for rule in ("min_fp_at_baseline_recall", "max_f1", "max_f2"):
    row = select_trial(TRIALS, baseline_tune, rule)
    picks[rule] = {name: row[name] for name in SEARCH_SPACE}
    picks[rule]["min_mask_objects_px"] = int(picks[rule]["min_mask_objects_px"])
    picks[rule]["max_candidates"] = int(picks[rule]["max_candidates"])

comparison = []
UNIQUE_BY_PICK = {"baseline": BASELINE_UNIQUE}
for name, params in picks.items():
    if name not in UNIQUE_BY_PICK:
        UNIQUE_BY_PICK[name] = evaluate(params, OK_IDS)[1]
    for split, ids in SPLITS.items():
        comparison.append({"config": name, "split": split, **score_candidates(UNIQUE_BY_PICK[name], ids)})
COMPARISON = pd.DataFrame(comparison)
COMPARISON.to_csv(OUTPUT_DIR / "comparison.csv", index=False)

view = COMPARISON.assign(
    precision=lambda d: (100 * d["precision"]).round(1),
    recall=lambda d: (100 * d["recall"]).round(1),
    f1=lambda d: (100 * d["f1"]).round(1),
    f2=lambda d: (100 * d["f2"]).round(1),
    candidates_per_tp=lambda d: d["candidates_per_tp"].round(1),
)[["config", "split", "unique_candidates", "tp", "fp", "precision", "sites_found", "sites_total",
   "recall", "f1", "f2", "candidates_per_tp"]]
display(view)

chosen = picks[SELECTION_RULE]
base_hold = score_candidates(BASELINE_UNIQUE, HOLDOUT_IDS)
chosen_hold = score_candidates(UNIQUE_BY_PICK[SELECTION_RULE], HOLDOUT_IDS)
print(f"\nSelected by {SELECTION_RULE!r}. Hold-out vs baseline: "
      f"sites {chosen_hold['sites_found']}/{chosen_hold['sites_total']} vs {base_hold['sites_found']}, "
      f"FP {chosen_hold['fp']} vs {base_hold['fp']}")
if all(chosen[name] == BASELINE_PARAMS[name] for name in SEARCH_SPACE):
    print("No tested configuration beat the baseline on the tuning split; run more trials.")
elif chosen_hold["sites_found"] >= base_hold["sites_found"] and chosen_hold["fp"] < base_hold["fp"]:
    print("The selected values generalise: fewer false positives without losing hold-out sites.")
else:
    print("The selected values do NOT beat the baseline on hold-out; treat them as overfit to the tuning split.")

print("\nChanged variables (baseline -> selected):")
for name in SEARCH_SPACE:
    if chosen[name] != BASELINE_PARAMS[name]:
        print(f"  {name:>22}: {BASELINE_PARAMS[name]} -> {chosen[name]}")

new_knobs = [
    name for name, neutral in (("optical_weight", 0.5), ("min_severity", 0.0), ("score_mode", "area_x_severity"))
    if chosen[name] != neutral
]
recommended = {
    "selection_rule": SELECTION_RULE,
    "raw_commit_sha": RAW_COMMIT_SHA,
    "match_iou": MATCH_IOU,
    "dedup_iou": DEDUP_IOU,
    "tune_incidents": TUNE_IDS,
    "holdout_incidents": HOLDOUT_IDS,
    "params": dict(chosen),
    "settings_for_ablation_notebooks": {
        key: chosen[key]
        for key in SEARCH_SPACE if key in (
            "min_slope_deg", "target_superpixel_m", "slic_compactness", "high_percentile", "low_percentile",
            "alpha", "min_blob_area_m2", "max_blob_area_m2", "max_elongation", "min_mask_objects_px",
            "max_candidates",
        )
    },
    "requires_pipeline_code_change": new_knobs,
    "metrics": {row["config"] + "/" + row["split"]: row for row in comparison},
}
(OUTPUT_DIR / "recommended_settings.json").write_text(
    json.dumps(plain(recommended), indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)
if new_knobs:
    print(f"\nNote: {new_knobs} are not in the ablation notebooks yet; porting them requires a code change.")

# Candidates of the selected configuration, for manual labelling of anything that is not ground truth.
with (OUTPUT_DIR / "selected_candidates.csv").open("w", newline="", encoding="utf-8") as handle:
    writer = csv.writer(handle)
    writer.writerow([
        "incident", "rank", "split", "matched_sites", "severity", "area_m2", "elongation", "score",
        "lon_min", "lat_min", "lon_max", "lat_max",
    ])
    holdout = set(HOLDOUT_IDS)
    for candidate in UNIQUE_BY_PICK[SELECTION_RULE]:
        writer.writerow([
            candidate["incident"], candidate["rank"], "holdout" if candidate["incident"] in holdout else "tune",
            ";".join(map(str, candidate["sites"])), candidate["severity"], candidate["area_m2"],
            candidate["elongation"], candidate["score"], *candidate["box"],
        ])
if _POOL is not None:
    _POOL.close()
    _POOL.join()
    _POOL = None
print(f"\nWrote {sorted(path.name for path in OUTPUT_DIR.iterdir())} to {OUTPUT_DIR}")

,config,split,unique_candidates,tp,fp,precision,sites_found,sites_total,recall,f1,f2,candidates_per_tp
0,baseline,tune,654,31,623,4.7,27,44,61.4,8.8,18.1,21.1
1,baseline,holdout,430,12,418,2.8,12,22,54.5,5.3,11.6,35.8
2,baseline,all,1084,43,1041,4.0,39,66,59.1,7.4,15.6,25.2
3,min_fp_at_baseline_recall,tune,422,31,391,7.3,28,44,63.6,13.2,25.1,13.6
4,min_fp_at_baseline_recall,holdout,295,3,292,1.0,3,22,13.6,1.9,3.9,98.3
5,min_fp_at_baseline_recall,all,717,34,683,4.7,31,66,47.0,8.6,16.9,21.1
6,max_f1,tune,110,17,93,15.5,15,44,34.1,21.3,27.5,6.5
7,max_f1,holdout,92,2,90,2.2,2,22,9.1,3.5,5.6,46.0
8,max_f1,all,202,19,183,9.4,17,66,25.8,13.8,19.1,10.6
9,max_f2,tune,205,23,182,11.2,22,44,50.0,18.3,29.6,8.9



Selected by 'min_fp_at_baseline_recall'. Hold-out vs baseline: sites 3/22 vs 12, FP 292 vs 418
The selected values do NOT beat the baseline on hold-out; treat them as overfit to the tuning split.

Changed variables (baseline -> selected):
          optical_weight: 0.5 -> 0.7
           min_slope_deg: 20.0 -> 25.0
     target_superpixel_m: 30.0 -> 20.0
        slic_compactness: 0.2 -> 0.1
         high_percentile: 99.7 -> 98.10765197970329
          low_percentile: 93.0 -> 82.07256701869453
                   alpha: 0.05 -> 0.00018632725274204788
        min_blob_area_m2: 3000.0 -> 2761.0036121034473
        max_blob_area_m2: 2000000.0 -> 490876.5237934569
          max_elongation: 5.0 -> 3.7564951992203244
     min_mask_objects_px: 5 -> 1
          max_candidates: 100 -> 33
            min_severity: 0.0 -> 0.08508780656398558

Note: ['optical_weight', 'min_severity'] are not in the ablation notebooks yet; porting them requires a code change.

Wrote ['comparison.csv', 'evidence_status.

## Reading the results

- `comparison.csv` – baseline vs the pick of each selection rule on the tuning, hold-out, and full sets. Trust the **hold-out** rows.
- `pareto.csv` / `trials.csv` – every trial on the tuning split; the Pareto front shows the recall/false-positive trade-off.
- `recommended_settings.json` – selected values, the subset that maps directly onto the ablation notebooks' `SETTINGS`, and any new knobs (`optical_weight`, `min_severity`, `score_mode`) that would require a pipeline code change.
- `selected_candidates.csv` – every unique candidate of the selected configuration. Rows without `matched_sites` are counted as false positives, but the 66 sites are **not exhaustive**: label these candidates and add confirmed landslides to the ground truth before trusting precision.

Caveats: ground truth is built from what the six variants detected, so variables that find landslides none of them found are not rewarded. With only ~22 hold-out sites, small differences (±1–2 sites) are within noise; prefer configurations that win on both splits.
